# Résoudre `4_precise` avec OR-Tools
Quatrième jeu de données : 8 000 modèles et 40 000 datasets. Son nom annonce la difficulté : les intervalles de besoins sont **très étroits**, de 0 à 10 unités. Il faut composer des tailles **presque exactes** avec des datasets.

On reprend le pipeline des notebooks 11 et 12 : modèle mathématique → SCIP → attribution des datasets → réparation. Et comme toujours, on commence par regarder ce qui change.

**Prérequis** : `datasets/4_precise.json`, `test_solution.py`, `pandas`, `ortools`.

**Durée d'exécution** : environ 20 minutes (dont 15 pour le solveur). Pour un essai rapide, mettez `SOLVER_SECONDS = 300` dans la partie 5 : on obtient environ 1 031 890.

**Version 2** : après le notebook 14, l'énergie est **mise à l'échelle** (exprimée en milliards) pour éviter les erreurs numériques du solveur, le solveur tourne 15 minutes au lieu de 5, et on essaie 40 ordres d'attribution au lieu de 10.

## 1. Charger l'instance et le repère

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import sys, json, time, bisect, random
import pandas as pd
from ortools.linear_solver import pywraplp
pd.set_option("display.width", 150)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from test_solution import getSolutionScore

DATASET = ROOT_DIR / "datasets" / "4_precise.json"
DATASET_TXT = DATASET.read_text()
instance = json.loads(DATASET_TXT)
energy_cap = instance["energyCap"]
print("commentaire de l'instance :", instance["comment"])

VALUE  = {m["id"]: m["value"] for m in instance["models"]}
ENERGY = {m["id"]: m["energyCost"] for m in instance["models"]}
NEEDS  = {m["id"]: [(t, b["lowerBound"], b["upperBound"]) for t, b in m["dataRequirements"].items()] for m in instance["models"]}
SIZE   = {d["id"]: d["size"] for d in instance["datasets"]}
TYPE   = {d["id"]: d["dataType"] for d in instance["datasets"]}
COPY   = {d["id"]: d["isCopyrighted"] for d in instance["datasets"]}
IMPOSSIBLE = {m for m, needs in NEEDS.items() if any(lo > hi for _, lo, hi in needs)}
LOWER = {m: {t: lo for t, lo, hi in needs if lo <= hi} for m, needs in NEEDS.items()}
MONO = {m: needs[0][0] for m, needs in NEEDS.items() if len(needs) == 1 and m not in IMPOSSIBLE}
FREE_SUPPLY  = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t and COPY[d] == 0) for t in "ntic"}
TOTAL_SUPPLY = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t) for t in "ntic"}

WINNER = 1_031_941
print(f"{len(VALUE)} modèles, {len(SIZE)} datasets | gagnant du concours : {WINNER:,}")

commentaire de l'instance : Those models are demanding!
8000 modèles, 40000 datasets | gagnant du concours : 1,031,941


## 2. Ce qui change par rapport à `2_medium` et `3_free`

In [2]:
print("nombre de besoins par modèle :", dict(sorted(Counter(len(n) for n in NEEDS.values()).items())))
print(f"modèles à un seul besoin (sources possibles) : {len(MONO)}")
print(f"modèles impossibles : {len(IMPOSSIBLE)}")
print(f"datasets sous droits : {sum(COPY.values()) / len(COPY):.0%}")
print(f"énergie de tous les modèles : {sum(ENERGY.values()) / energy_cap:.2f} × le plafond")
widths = Counter(hi - lo for needs in NEEDS.values() for t, lo, hi in needs if lo <= hi)
print("largeur des intervalles [lower, upper] :", dict(sorted(widths.items())))

nombre de besoins par modèle : {2: 3206, 3: 3196, 4: 1598}
modèles à un seul besoin (sources possibles) : 0
modèles impossibles : 380
datasets sous droits : 50%
énergie de tous les modèles : 5.96 × le plafond
largeur des intervalles [lower, upper] : {0: 1102, 1: 2121, 2: 2153, 3: 2137, 4: 2194, 5: 2177, 6: 2244, 7: 2152, 8: 2135, 9: 2197, 10: 1106}


**Lecture**
- **Aucun modèle n'a un seul besoin** : aucun ne peut servir de source. Les 380 modèles impossibles sont donc **définitivement exclus**. Les sources, qui ont compté sur `2_medium` et `3_free`, disparaissent du problème.
- La moitié des datasets est sous droits : la rareté des données libres revient, comme sur `2_medium`.
- Les intervalles font de **0 à 10 unités** de large, pour des besoins d'environ 17 000. Plus de 1 000 besoins exigent une taille **exacte**.

## 3. Composer une taille précise
On reprend `fill` avec la recherche de paire du notebook 11. La paire est ici essentielle : pour finir un remplissage dans une fenêtre de quelques unités, un seul dataset de la bonne taille est rare, mais **deux** datasets dont la somme tombe juste sont nombreux. Avec 5 000 datasets d'un type, il y a environ 12 millions de paires.

In [3]:
class Pool:
    """Réserve de datasets d'un type et d'un statut, triée par taille."""
    def __init__(self, ids):
        self.ids = sorted(ids, key=SIZE.get)
        self.sizes = [SIZE[d] for d in self.ids]
    def take(self, i):
        self.sizes.pop(i)
        return self.ids.pop(i)
    def put(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        self.sizes.insert(i, SIZE[d]); self.ids.insert(i, d)
    def remove(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        while self.ids[i] != d:
            i += 1
        self.take(i)

def find_pair(sizes, a, b):
    """Deux positions i < j avec sizes[i] + sizes[j] dans [a, b], somme la plus petite. Deux pointeurs : O(n)."""
    i, j, best = 0, bisect.bisect_right(sizes, b) - 1, None
    while i < j:
        s = sizes[i] + sizes[j]
        if s > b:
            j -= 1
        elif s < a:
            i += 1
        else:
            if best is None or s < best[0]:
                best = (s, i, j)
            j -= 1
    return best

def fill(pool, lower, upper):
    """Prend des datasets dont la somme tombe dans [lower, upper]. Rend la liste, ou None (rien n'est pris)."""
    taken, total = [], 0
    while total < lower:
        a, b = lower - total, upper - total
        i = bisect.bisect_left(pool.sizes, a)
        if i < len(pool.sizes) and pool.sizes[i] <= b:          # 1. un dataset termine
            total += pool.sizes[i]; taken.append(pool.take(i)); continue
        pair = find_pair(pool.sizes, a, b) if pool.sizes and a <= 2 * pool.sizes[-1] else None
        if pair:                                                 # 2. une paire termine
            _, i, j = pair
            total += pool.sizes[i] + pool.sizes[j]
            taken.append(pool.take(j)); taken.append(pool.take(i)); continue
        j = bisect.bisect_right(pool.sizes, b) - 1               # 3. le plus gros qui rentre
        if j < 0:
            break
        total += pool.sizes[j]; taken.append(pool.take(j))
    if lower <= total <= upper:
        return taken
    for d in taken:
        pool.put(d)
    return None

# Exemple : un besoin de largeur 0
m, (t, lo, hi) = next((m, n) for m, needs in NEEDS.items() for n in needs if n[1] == n[2])
pool = Pool([d for d in SIZE if TYPE[d] == t])
got = fill(pool, lo, hi)
print(f"modèle {m}, besoin {t} = exactement {lo} -> tailles {[SIZE[d] for d in got]}, somme {sum(SIZE[d] for d in got)}")

modèle 4, besoin n = exactement 11641 -> tailles [10000, 1641], somme 11641


Chaque besoin, **pris isolément**, peut-il être rempli ? On essaie tous les besoins un par un, en remettant les datasets après chaque essai (une vingtaine de secondes).

In [4]:
t0 = time.time()
pools = {t: Pool([d for d in SIZE if TYPE[d] == t]) for t in "ntic"}
n_needs, failures, n_datasets = 0, [], Counter()
for m, needs in NEEDS.items():
    for t, lo, hi in needs:
        if lo > hi:
            continue
        n_needs += 1
        got = fill(pools[t], lo, hi)
        if got is None:
            failures.append((m, t, lo, hi))
        else:
            n_datasets[len(got)] += 1
            for d in got:
                pools[t].put(d)
print(f"{n_needs} besoins testés en {time.time() - t0:.0f} s | irréalisables : {len(failures)}")
print("nombre de datasets par besoin :", dict(sorted(n_datasets.items())))

21718 besoins testés en 23 s | irréalisables : 0
nombre de datasets par besoin : {1: 5622, 2: 7177, 3: 5420, 4: 2274, 5: 1090, 6: 135}


**Lecture** : **tous** les besoins sont réalisables isolément, en 1 à 6 datasets. La précision n'est donc pas un obstacle en soi. La difficulté viendra de la **concurrence** : quand des centaines de modèles puisent dans les mêmes réserves, les bonnes tailles s'épuisent.

## 4. Le modèle mathématique, et un bug corrigé
Même modèle que dans le notebook 11 : $x_i$ (pleine valeur, données libres), $y_i$ (demi-valeur), $z_m$ et $u_{s,m,t}$ (sources pour les modèles impossibles).

**Correction importante.** Dans les notebooks 11 et 12, la contrainte « une source par besoin impossible » n'était créée que pour les besoins qui avaient **au moins une source candidate**. Ici, aucun modèle ne peut être source. La contrainte n'était donc jamais créée, et $z_m$ restait libre : le solveur entraînait gratuitement 173 modèles impossibles, pour un objectif de 1 035 477, bien au-dessus du gagnant. Cela aurait dû alerter : **un modèle qui bat largement le meilleur score connu cache souvent une contrainte oubliée**.

La correction : pour chaque besoin impossible, on écrit $\sum_s u_{s,m,t} = z_m$ **même si la somme est vide**, ce qui force $z_m = 0$. Les résultats des notebooks 11 et 12 ne sont pas touchés : sur ces instances, chaque type avait des sources candidates.

**Mise à l'échelle de l'énergie.** Dans le notebook 14, des coefficients d'énergie de $10^{12}$ ont fait annoncer au solveur un faux optimum. On divise donc l'énergie par $10^9$ dans le modèle. Le problème mathématique est exactement le même, mais les calculs en virgule flottante sont plus sûrs. On vérifie ensuite l'énergie **exacte**, en nombres entiers, sur la solution.

In [5]:
def solve_model(seconds=300):
    t0 = time.time()
    S = pywraplp.Solver.CreateSolver("SCIP")
    x = {m: S.BoolVar(f"x{m}") for m in VALUE if m not in IMPOSSIBLE}
    y = {m: S.BoolVar(f"y{m}") for m in VALUE if m not in IMPOSSIBLE}
    z = {m: S.BoolVar(f"z{m}") for m in IMPOSSIBLE}
    u = {(s, m, t): S.BoolVar(f"u{s}_{m}_{t}")
         for m in IMPOSSIBLE for t, lo, hi in NEEDS[m] if lo > hi
         for s, ts in MONO.items() if ts == t}
    as_source, sources_of = defaultdict(list), defaultdict(list)
    for (s, m, t), var in u.items():
        as_source[s].append(var)
        sources_of[(m, t)].append(var)

    for m in x:
        S.Add(x[m] + y[m] + sum(as_source[m]) <= 1)                              # un seul rôle
    for m in IMPOSSIBLE:                                                           # CORRIGÉ : même sans candidat
        for t, lo, hi in NEEDS[m]:
            if lo > hi:
                S.Add(sum(sources_of[(m, t)]) == z[m])
    E = 1e9                                       # énergie en milliards : coefficients ~10^3 au lieu de ~10^12
    S.Add(sum(ENERGY[m] / E * (x[m] + y[m]) for m in x) + sum(ENERGY[m] / E * z[m] for m in z)
          + sum(ENERGY[s] / E * var for (s, m, t), var in u.items()) <= energy_cap / E)      # énergie
    for t in "ntic":
        S.Add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m]) <= FREE_SUPPLY[t])        # données libres
        S.Add(sum(LOWER[m][t] * (x[m] + y[m]) for m in x if t in LOWER[m])                    # données totales
              + sum(LOWER[m][t] * z[m] for m in z if t in LOWER[m])
              + sum(LOWER[s][t] * var for (s, m, tt), var in u.items() if tt == t) <= TOTAL_SUPPLY[t])
    S.Maximize(sum(VALUE[m] * x[m] + (VALUE[m] // 2) * y[m] for m in x) + sum((VALUE[m] // 2) * z[m] for m in z))

    S.SetTimeLimit(seconds * 1000)
    params = pywraplp.MPSolverParameters()
    params.SetDoubleParam(params.RELATIVE_MIP_GAP, 0.0)                           # leçon du notebook 12
    status = S.Solve(params)
    val = lambda v: round(v.solution_value())
    sol = dict(full=[m for m in x if val(x[m])], half=[m for m in x if val(y[m])],
               impossible=[m for m in z if val(z[m])], sources=[(s, m, t) for (s, m, t), var in u.items() if val(var)])
    obj, bound = S.Objective().Value(), S.Objective().BestBound()
    info = dict(statut="OPTIMAL" if status == pywraplp.Solver.OPTIMAL else "FEASIBLE", objectif=round(obj),
                borne=round(bound, 1), écart_prouvé=round(bound - obj, 1), secondes=round(time.time() - t0),
                pleins=len(sol["full"]), demi=len(sol["half"]), impossibles=len(sol["impossible"]))
    return sol, info

## 5. Résoudre
On laisse SCIP travailler 15 minutes, en exigeant un écart nul (leçon du notebook 12).

In [6]:
SOLVER_SECONDS = 900
sol, info = solve_model(seconds=SOLVER_SECONDS)
used = sum(ENERGY[m] for m in sol["full"] + sol["half"])
print(f"énergie exacte : {used:,} <= {energy_cap:,} ? {used <= energy_cap}")
pd.DataFrame([info], index=["SCIP"])

énergie exacte : 677,989,112,582,982 <= 677,991,728,790,302 ? True


,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,impossibles
SCIP,FEASIBLE,1031913,1031946.9,33.9,901,2178,190,0


**Lecture**
- La **borne** est d'environ 1 031 947 : le gagnant (1 031 941) est donc à **6 points au plus** de l'optimum du modèle.
- En 15 minutes, SCIP ne prouve pas l'optimum. Selon l'exécution, il s'arrête entre 1 031 913 et 1 031 935, à 10 à 35 points de la borne. Le problème est gros (8 000 modèles) et serré.
- **Avec une limite de temps, le résultat n'est pas reproductible d'une exécution à l'autre** : en 15 minutes, le solveur explore plus ou moins de nœuds selon la vitesse et la charge de la machine.
- Comparaison avec la version 1 (énergie non mise à l'échelle) : 1 031 901 en 5 minutes et 1 031 922 en 15 minutes, contre 1 031 910 et 1 031 913 à 1 031 935 avec la mise à l'échelle. **La mise à l'échelle aide un peu, sans plus** : sur ce dataset, la difficulté vient surtout de la taille de la recherche, pas des erreurs numériques.
- Aucun modèle impossible n'est entraîné : la correction fonctionne.

## 6. Attribuer les datasets
On combine les deux notebooks précédents :
- du **notebook 12**, la file d'attente unique pour tous les modèles (ici sans sources) ;
- du **notebook 11**, les deux statuts : données libres seulement pour les modèles pleins, données sous droits d'abord pour les autres.

Les modèles pleins qu'on ne peut pas servir en données libres repassent ensuite à demi-valeur.

In [7]:
def give(pools, model, needs, statuses):
    got = []
    for t, lo, hi in needs:
        res = None
        for s in statuses:
            res = fill(pools[(t, s)], lo, hi)
            if res is not None:
                break
        if res is None:
            for d in got:
                pools[(TYPE[d], COPY[d])].put(d)
            return None
        got += res
    return [[int(d), int(model)] for d in got]

def pack(sol, order_key):
    """File unique triée par order_key(modèle). Rend (plan, modèles pleins ratés)."""
    pools = {(t, c): Pool([d for d in SIZE if TYPE[d] == t and COPY[d] == c]) for t in "ntic" for c in (0, 1)}
    jobs = [(m, (0,)) for m in sol["full"]] + [(m, (1, 0)) for m in sol["half"]]
    data, failed = {}, []
    for m, statuses in sorted(jobs, key=lambda job: order_key(job[0])):
        got = give(pools, m, NEEDS[m], statuses)
        if got is None:
            failed.append(m)
        else:
            data[m] = got
    for m in failed:                                       # seconde chance, à demi-valeur
        got = give(pools, m, NEEDS[m], (1, 0))
        if got is not None:
            data[m] = got
    return {"dataMappings": [p for got in data.values() for p in got], "modelMappings": []}, failed

### La réparation du notebook 11, élargie
Dans le notebook 11, la réparation essayait d'échanger un modèle à demi-valeur avec **au plus 40** modèles pleins. Ici, cette limite empêchait souvent de réparer. On essaie maintenant **tous** les partenaires possibles. Cela reste rapide (quelques secondes), car on s'arrête au premier échange réussi.

In [8]:
class State:
    """Solution modifiable, avec journal d'annulation (notebook 10)."""
    def __init__(self, plan):
        self.data = {}
        for d, m in plan["dataMappings"]:
            self.data.setdefault(m, []).append(d)
        used = {d for d, _ in plan["dataMappings"]}
        self.pools = {(t, c): Pool([d for d in SIZE if d not in used and TYPE[d] == t and COPY[d] == c])
                      for t in "ntic" for c in (0, 1)}
        self.full = {m: not any(COPY[d] for d in ds) for m, ds in self.data.items()}
        self.energy = sum(ENERGY[m] for m in self.data)
        self.log = []

    def remove(self, m):
        ds, full = self.data.pop(m), self.full.pop(m)
        for d in ds:
            self.pools[(TYPE[d], COPY[d])].put(d)
        self.energy -= ENERGY[m]
        self.log.append(("remove", m, ds, full))

    def add(self, m, statuses):
        if self.energy + ENERGY[m] > energy_cap:
            return False
        got = give(self.pools, m, NEEDS[m], statuses)
        if got is None:
            return False
        self.data[m] = [d for d, _ in got]
        self.full[m] = not any(COPY[d] for d in self.data[m])
        self.energy += ENERGY[m]
        self.log.append(("add", m, None, None))
        return True

    def undo(self):
        while self.log:
            op, m, ds, full = self.log.pop()
            if op == "add":
                for d in self.data.pop(m):
                    self.pools[(TYPE[d], COPY[d])].put(d)
                self.full.pop(m); self.energy -= ENERGY[m]
            else:
                for d in ds:
                    self.pools[(TYPE[d], COPY[d])].remove(d)
                self.data[m], self.full[m] = ds, full
                self.energy += ENERGY[m]

    def plan(self):
        return {"dataMappings": [[int(d), int(m)] for m, ds in self.data.items() for d in ds], "modelMappings": []}

TYPES_OF = {m: {t for t, _, _ in needs} for m, needs in NEEDS.items()}
gain_full = lambda m: VALUE[m] - VALUE[m] // 2

def repair(plan):
    s = State(plan)
    n_repairs, improved = 0, True
    while improved:
        improved = False
        for b in sorted([m for m in s.data if not s.full[m]], key=gain_full, reverse=True):
            if b not in s.data or s.full[b]:
                continue
            s.log = []; s.remove(b)
            if s.add(b, (0,)):
                n_repairs += 1; improved = True; continue
            s.undo()
            partners = sorted([a for a in s.data if s.full[a] and TYPES_OF[a] & TYPES_OF[b] and gain_full(a) < gain_full(b)],
                              key=gain_full)
            for a in partners:                                     # TOUS les partenaires (40 dans le notebook 11)
                s.log = []; s.remove(a); s.remove(b)
                if s.add(b, (0,)) and s.add(a, (1, 0)):
                    n_repairs += 1; improved = True; break
                s.undo()
    return s.plan(), n_repairs

### Plusieurs ordres d'attribution
Sur `2_medium`, l'ordre « petits besoins d'abord » et l'ordre « plus contraints d'abord » gagnaient chacun selon les cas. Ici, on les essaie tous les deux, plus **40 ordres bruités** : l'ordre « petits besoins », avec un facteur aléatoire de ±20 % sur chaque modèle. On garde le meilleur résultat. C'est une **recherche à départs multiples** : un algorithme glouton déterministe donne toujours la même réponse, et un peu de hasard permet d'explorer plusieurs réponses.

In [9]:
orders = {"petits besoins": lambda m: sum(LOWER[m].values()),
          "plus contraints": lambda m: (min(hi for t, lo, hi in NEEDS[m] if lo <= hi), sum(LOWER[m].values()))}
rng = random.Random(0)
for k in range(40):
    noise = {m: rng.uniform(0.8, 1.2) for m in NEEDS}
    orders[f"bruité {k}"] = lambda m, nz=noise: sum(LOWER[m].values()) * nz[m]

rows, best = [], (0, None, None)
t0 = time.time()
for name, key in orders.items():
    plan0, failed = pack(sol, key)
    score0 = getSolutionScore(json.dumps(plan0), DATASET_TXT, show_details=False)[0]
    plan, n = repair(plan0)
    score = getSolutionScore(json.dumps(plan), DATASET_TXT, show_details=False)[0]
    rows.append(dict(ordre=name, ratés=len(failed), avant_réparation=score0, réparations=n, score=score, écart_au_gagnant=score - WINNER))
    if score > best[0]:
        best = (score, name, plan)
print(f"{len(orders)} ordres en {time.time() - t0:.0f} s")
orders_table = pd.DataFrame(rows).set_index("ordre")
orders_table

42 ordres en 136 s


,ratés,avant_réparation,réparations,score,écart_au_gagnant
ordre,,,,,
petits besoins,2,1030922,4,1031881,-60
plus contraints,2,1030965,3,1031884,-57
bruité 0,2,1030940,3,1031872,-69
bruité 1,2,1030948,4,1031866,-75
bruité 2,1,1031428,2,1031876,-65
bruité 3,2,1030943,4,1031881,-60
bruité 4,1,1031428,2,1031852,-89
bruité 5,2,1030940,5,1031859,-82
bruité 6,1,1031434,2,1031891,-50


**Lecture**
- Selon l'ordre, un ou deux modèles pleins ne trouvent pas leurs données libres. Ce sont souvent des modèles précieux, et chacun coûte jusqu'à 500 points avant réparation.
- La réparation récupère presque tout, mais pas toujours : le résultat final varie de 75 points selon l'ordre (de 1 031 816 à 1 031 891 dans cette exécution). **L'attribution des datasets est une loterie**, et plusieurs tirages valent mieux qu'un seul.

## 7. Comparaison et vérification

In [10]:
best_score, best_name, best_plan = best
summary = pd.DataFrame({
    "score": [best_score, info["objectif"], WINNER, info["borne"]],
}, index=[f"meilleur plan (ordre « {best_name} »)", "sélection du solveur (avant datasets)", "gagnant du concours", "borne du solveur"])
summary["écart_au_gagnant"] = summary.score - WINNER
summary["part_du_gagnant"] = (summary.score / WINNER).map("{:.4%}".format)
display(summary)

score, valid, message = getSolutionScore(json.dumps(best_plan), DATASET_TXT, show_details=True)
print("score officiel =", score, "| valide =", valid, message)

,score,écart_au_gagnant,part_du_gagnant
meilleur plan (ordre « bruité 6 »),1031891.0,-50.0,99.9952%
sélection du solveur (avant datasets),1031913.0,-28.0,99.9973%
gagnant du concours,1031941.0,0.0,100.0000%
borne du solveur,1031946.9,5.9,100.0006%



Total value: 1031891
Energy used: 677989112582982 / 677991728790302 (100.00%)
Number of used models: 2368 / 8000 (29.60%)
  Number of devalued models: 191 (8.07%)
  Number of models used as data source: 0 (0.00%)
Number of used datasets: 20587 / 40000 (51.47%)
score officiel = 1031891 | valide = True 


**Lecture**
- Dans cette exécution, on arrive à 50 points du gagnant (99,995 %). Lors d'une exécution précédente, avec une sélection du solveur à 1 031 935, le meilleur des 40 ordres a donné **1 031 910**, à 31 points du gagnant (99,997 %). C'est cette solution qui est dans `4_precise_main.json` : la partie 8 ne remplace la solution que si elle l'améliore.
- L'écart se partage entre deux sources : la sélection du solveur (entre 10 et 35 points sous la borne selon l'exécution) et l'attribution des datasets, qui perd encore une vingtaine de points. Une meilleure sélection n'est d'ailleurs pas toujours plus facile à découper : c'est pourquoi on essaie beaucoup d'ordres.
- Pistes essayées sans succès : le recuit du notebook 10 en finition (aucun gain), et la boucle du notebook 11, qui interdit la pleine valeur aux modèles qui ne trouvent pas leurs données libres. Ici, la boucle empire les choses : à chaque itération, **d'autres** modèles échouent, et la borne baisse (1 031 891, puis 1 031 829). Les échecs ne viennent pas de modèles particuliers, mais de la concurrence entre tous.

## 8. Écrire le fichier de solution
On ne remplace la solution principale que si on l'améliore (la sortie de cette cellule a été effacée après correction du code).

In [ ]:
out_dir = ROOT_DIR / "solutions"; out_dir.mkdir(exist_ok=True)
out = out_dir / "4_precise_main.json"
current = getSolutionScore(out.read_text(), DATASET_TXT, show_details=False)[0] if out.exists() else 0
if valid and score > current:
    out.write_text(json.dumps(best_plan))
    print(f"4_precise_main.json mis à jour : {current:,} -> {score:,}")
else:
    print(f"4_precise_main.json inchangé ({current:,})")

## À retenir
1. **Regarder la structure avant de modéliser.** Ici, aucun modèle n'a un seul besoin : les sources disparaissent, et les 380 modèles impossibles sont perdus d'avance.
2. **Un score trop beau cache une contrainte oubliée.** Le modèle annonçait 3 500 points de plus que le gagnant. La cause était une contrainte jamais créée quand la liste de candidats était vide. C'est un bug classique : **une boucle sur une liste vide n'écrit aucune contrainte**.
3. **La précision n'est pas le vrai problème, la concurrence l'est.** Chaque besoin, seul, se remplit facilement grâce aux paires de datasets. C'est le partage des réserves entre des milliers de modèles qui coûte des points.
4. **Plusieurs départs valent mieux qu'un.** Un ordre d'attribution est un pari. En essayer une douzaine, avec un peu de hasard, fait gagner des dizaines de points en moins d'une minute.
5. **Une réparation trop timide ne répare pas.** Limiter la recherche de partenaires à 40 suffisait sur `2_medium`, pas ici. Une limite de calcul fixée sur un cas doit être revérifiée sur le suivant.